In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/cleaned_aqi_final.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print("Original shape:", df.shape)

# =========================
# FEATURE ENGINEERING
# =========================
df['AQI_lag1'] = df['AQI'].shift(1)
df['AQI_lag3'] = df['AQI'].shift(3)
df['AQI_lag7'] = df['AQI'].shift(7)

df['PM25_roll7'] = df['PM2.5'].rolling(7).mean()
df['PM10_roll7'] = df['PM10'].rolling(7).mean()

df = df.dropna()

print("After engineering:", df.shape)

# =========================
# AQI CATEGORY
# =========================
def get_category(aqi):
    if aqi <= 50:
        return 0
    elif aqi <= 100:
        return 1
    elif aqi <= 200:
        return 2
    elif aqi <= 300:
        return 3
    elif aqi <= 400:
        return 4
    else:
        return 5

df['AQI_Class'] = df['AQI'].apply(get_category)

# =========================
# FEATURES
# =========================
features = [
    'PM2.5',
    'PM10',
    'NO',
    'NO2',
    'NOx',
    'NH3',
    'CO',
    'SO2',
    'O3',
    'Benzene',
    'Toluene',
    'AQI_lag1',
    'AQI_lag3',
    'AQI_lag7',
    'PM25_roll7',
    'PM10_roll7'
]

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[features])

# =========================
# CREATE SEQUENCES
# =========================
SEQ_LENGTH = 30

X_sequences = []
y_targets = []

for i in range(SEQ_LENGTH, len(scaled_features)):
    X_sequences.append(scaled_features[i-SEQ_LENGTH:i])
    y_targets.append(df['AQI_Class'].iloc[i])

X_sequences = np.array(X_sequences)
y_targets = np.array(y_targets)

print("X shape:", X_sequences.shape)
print("y shape:", y_targets.shape)

# =========================
# SAVE
# =========================
np.save("../data/X_class.npy", X_sequences)
np.save("../data/y_class.npy", y_targets)

print("Saved successfully.")

Original shape: (1827, 15)
After engineering: (1820, 20)
X shape: (1790, 30, 16)
y shape: (1790,)
Saved successfully.
